# LSC Post-Hoc Contributor Diagnostics

This notebook builds descriptive post-hoc diagnostics for the dissertation's sentiment, intensity, and breadth results. It uses completed LSC outputs and does not rerun Common Crawl processing, VAD matching, frame classification, or embedding models.

The outputs are intended as interpretive diagnostics rather than new inferential tests.

## Setup

The analysis uses three broad publication-year periods: 2014-2017, 2018-2021, and 2022-2026. Target outputs are restricted to ADHD and Autism, with Overall, Clinical, and Lived experience strata.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import re

import numpy as np
import pandas as pd
import spacy
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "paper" / "main.tex").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate project root from the current working directory.")


PROJECT_ROOT = find_project_root()

VAD_MATCH_PATH = PROJECT_ROOT / "data/interim/lsc/vad/lsc_vad_collocate_matches.parquet"
BREADTH_CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/breadth/lsc_breadth_sampled_contexts.parquet"
BREADTH_EMBEDDING_PATH = PROJECT_ROOT / "data/interim/lsc/breadth/lsc_breadth_embeddings_normalised.npy"
BREADTH_ANNUAL_PATH = PROJECT_ROOT / "data/processed/lsc/breadth/lsc_breadth_annual_scores.csv"
SENTIMENT_ANNUAL_PATH = PROJECT_ROOT / "data/processed/lsc/sentiment/lsc_sentiment_annual_valence.csv"
INTENSITY_ANNUAL_PATH = PROJECT_ROOT / "data/processed/lsc/intensity/lsc_intensity_annual_arousal.csv"

POSTHOC_DIR = PROJECT_ROOT / "data/processed/lsc/posthoc"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports/tables/lsc/posthoc"
for directory in [POSTHOC_DIR, REPORT_TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SENTIMENT_LATEX_PATH = REPORT_TABLE_DIR / "lsc_posthoc_sentiment_collocates.tex"
AROUSAL_LATEX_PATH = REPORT_TABLE_DIR / "lsc_posthoc_arousal_collocates.tex"
BREADTH_WORD_LATEX_PATH = REPORT_TABLE_DIR / "lsc_posthoc_breadth_content_words.tex"

TARGET_UNITS = ["ADHD", "Autism"]
TARGET_ORDER = {unit: i for i, unit in enumerate(TARGET_UNITS)}
REPORT_FRAMES = ["substantive_core_overall", "clinical_only", "lived_only"]
FRAME_LABELS = {
    "substantive_core_overall": "Overall",
    "clinical_only": "Clinical",
    "lived_only": "Lived experience",
}
FRAME_ORDER = {frame: i for i, frame in enumerate(REPORT_FRAMES)}

PERIODS = [
    {"period": "2014-2017", "period_start": 2014, "period_end": 2017, "period_order": 1},
    {"period": "2018-2021", "period_start": 2018, "period_end": 2021, "period_order": 2},
    {"period": "2022-2026", "period_start": 2022, "period_end": 2026, "period_order": 3},
]
PERIOD_TABLE = pd.DataFrame(PERIODS)
PERIOD_LABELS = PERIOD_TABLE["period"].tolist()
PERIOD_LOOKUP = {
    year: period["period"]
    for period in PERIODS
    for year in range(period["period_start"], period["period_end"] + 1)
}
PERIOD_ORDER = PERIOD_TABLE.set_index("period")["period_order"].to_dict()
PERIOD_START = PERIOD_TABLE.set_index("period")["period_start"].to_dict()
PERIOD_END = PERIOD_TABLE.set_index("period")["period_end"].to_dict()

CELL_COLUMNS = [
    "analysis_unit",
    "frame_stratum",
    "frame_label",
    "period",
    "period_order",
    "period_start",
    "period_end",
]
CELL_SORT_COLUMNS = ["analysis_order", "frame_order", "period_order"]
CONTENT_POS = {"ADJ", "ADV", "NOUN", "PROPN", "VERB"}
COLLOCATE_TOP_N = 5
REPORT_TOP_TERMS = 3
BREADTH_TOP_CONTEXTS_PER_CELL = 10
BREADTH_WORD_SOURCE_CONTEXTS_PER_CELL = 20
BREADTH_TOP_WORDS_PER_CELL = 10

TARGET_WORDS = {
    "ADHD": {"add", "adhd", "attention", "deficit", "hyperactivity", "hyperactive"},
    "Autism": {"autism", "autistic", "asd", "spectrum"},
}
BOILERPLATE_WORDS = {
    "http", "https", "www", "com", "org", "html", "copyright", "reserved", "rights",
    "cookie", "privacy", "policy", "website", "site", "page", "read", "more",
}
LATEX_REPLACEMENTS = {
    "\\": r"\textbackslash{}",
    "&": r"\&",
    "%": r"\%",
    "$": r"\$",
    "#": r"\#",
    "_": r"\_",
    "{": r"\{",
    "}": r"\}",
    "~": r"\textasciitilde{}",
    "^": r"\textasciicircum{}",
}


def add_period_columns(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    output["period"] = output["lsc_year"].map(PERIOD_LOOKUP)
    if output["period"].isna().any():
        bad_years = sorted(output.loc[output["period"].isna(), "lsc_year"].dropna().unique().tolist())
        raise ValueError(f"Rows fall outside the configured period bins: {bad_years}")
    output["period_order"] = output["period"].map(PERIOD_ORDER).astype(int)
    output["period_start"] = output["period"].map(PERIOD_START).astype(int)
    output["period_end"] = output["period"].map(PERIOD_END).astype(int)
    return output


def add_display_order(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    output["frame_label"] = output["frame_stratum"].map(FRAME_LABELS)
    output["analysis_order"] = output["analysis_unit"].map(TARGET_ORDER).astype(int)
    output["frame_order"] = output["frame_stratum"].map(FRAME_ORDER).astype(int)
    return output


def latex_escape(value: object) -> str:
    text = str(value if value is not None else "")
    return "".join(LATEX_REPLACEMENTS.get(char, char) for char in text)


def latex_cell_lines(lines: list[str]) -> str:
    escaped_lines = [latex_escape(line) for line in lines]
    return r"\begin{minipage}[t]{\linewidth}\raggedright " + r"\\ ".join(escaped_lines) + r"\end{minipage}"


def write_table(frame: pd.DataFrame, name: str) -> tuple[Path, Path]:
    data_path = POSTHOC_DIR / name
    report_path = REPORT_TABLE_DIR / name
    frame.to_csv(data_path, index=False)
    frame.to_csv(report_path, index=False)
    return data_path, report_path


def write_latex_table(content: str, path: Path) -> Path:
    path.write_text(content, encoding="utf-8")
    return path


for required_path in [
    VAD_MATCH_PATH,
    BREADTH_CONTEXT_PATH,
    BREADTH_EMBEDDING_PATH,
    BREADTH_ANNUAL_PATH,
    SENTIMENT_ANNUAL_PATH,
    INTENSITY_ANNUAL_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
print(f"Project root: {PROJECT_ROOT}")
print(f"Post-hoc outputs: {POSTHOC_DIR.relative_to(PROJECT_ROOT)}")


Project root: /Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak
Post-hoc outputs: data/processed/lsc/posthoc


## VAD Collocate Contributors

The VAD match table has already undergone tokenisation, POS filtering, stopword removal, and lemmatisation in the upstream sentiment and severity notebooks. These post-hoc tables therefore do not apply an additional content-word screen. They rank the preprocessed collocate units that directly enter the annual valence and arousal aggregates.


In [2]:
vad_matches = pd.read_parquet(VAD_MATCH_PATH)
vad_target = vad_matches.loc[
    vad_matches["analysis_unit"].isin(TARGET_UNITS)
    & vad_matches["frame_stratum"].isin(REPORT_FRAMES)
].copy()
vad_target = add_display_order(add_period_columns(vad_target))

expected_cells = {
    (unit, frame, period["period"])
    for unit in TARGET_UNITS
    for frame in REPORT_FRAMES
    for period in PERIODS
}
observed_vad_cells = set(
    vad_target[["analysis_unit", "frame_stratum", "period"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
missing_vad_cells = sorted(expected_cells - observed_vad_cells)
if missing_vad_cells:
    raise ValueError(f"Missing VAD cells: {missing_vad_cells}")

cell_totals = (
    vad_target.groupby(CELL_COLUMNS, as_index=False)
    .agg(
        total_matches_for_cell=("collocate", "size"),
        total_documents_for_cell=("doc_id", "nunique"),
        valence_mean_for_cell=("valence", "mean"),
        arousal_mean_for_cell=("arousal", "mean"),
    )
)

collocate_contributions = (
    vad_target.groupby([*CELL_COLUMNS, "collocate", "collocate_type"], as_index=False)
    .agg(
        occurrences=("collocate", "size"),
        documents=("doc_id", "nunique"),
        valence=("valence", "mean"),
        arousal=("arousal", "mean"),
        weighted_valence_contribution=("valence", "sum"),
        weighted_arousal_contribution=("arousal", "sum"),
    )
    .merge(cell_totals, on=CELL_COLUMNS, how="left")
)
collocate_contributions["match_share"] = (
    collocate_contributions["occurrences"] / collocate_contributions["total_matches_for_cell"]
)
collocate_contributions["valence_contribution_to_cell_mean"] = (
    collocate_contributions["weighted_valence_contribution"]
    / collocate_contributions["total_matches_for_cell"]
)
collocate_contributions["arousal_contribution_to_cell_mean"] = (
    collocate_contributions["weighted_arousal_contribution"]
    / collocate_contributions["total_matches_for_cell"]
)
collocate_contributions = add_display_order(collocate_contributions)


def collocate_token_set(value: object) -> set[str]:
    return set(re.findall(r"[a-z]+", str(value or "").lower()))


def target_vocabulary_flags(row: pd.Series) -> pd.Series:
    tokens = collocate_token_set(row["collocate"])
    same_target_words = TARGET_WORDS.get(row["analysis_unit"], set())
    other_target_words = set().union(
        *(words for unit, words in TARGET_WORDS.items() if unit != row["analysis_unit"])
    )
    return pd.Series(
        {
            "same_target_vocabulary_collocate": bool(tokens & same_target_words),
            "other_target_vocabulary_collocate": bool(tokens & other_target_words),
        }
    )


target_flags = collocate_contributions.apply(target_vocabulary_flags, axis=1)
collocate_contributions = pd.concat([collocate_contributions, target_flags], axis=1)


def top_signed_contributors(
    frame: pd.DataFrame,
    value_column: str,
    contribution_column: str,
    positive_label: str,
    negative_label: str,
    n: int = COLLOCATE_TOP_N,
) -> pd.DataFrame:
    sort_columns = ["analysis_order", "frame_order", "period_order", contribution_column]
    positive = (
        frame.loc[frame[contribution_column] > 0]
        .sort_values(sort_columns, ascending=[True, True, True, False])
        .groupby(CELL_COLUMNS, as_index=False)
        .head(n)
        .copy()
    )
    positive["contribution_direction"] = positive_label
    positive["rank"] = positive.groupby(CELL_COLUMNS).cumcount() + 1

    negative = (
        frame.loc[frame[contribution_column] < 0]
        .assign(abs_contribution=lambda data: data[contribution_column].abs())
        .sort_values(["analysis_order", "frame_order", "period_order", "abs_contribution"], ascending=[True, True, True, False])
        .groupby(CELL_COLUMNS, as_index=False)
        .head(n)
        .drop(columns="abs_contribution")
        .copy()
    )
    negative["contribution_direction"] = negative_label
    negative["rank"] = negative.groupby(CELL_COLUMNS).cumcount() + 1

    output = pd.concat([positive, negative], ignore_index=True)
    output = output.sort_values(
        ["analysis_order", "frame_order", "period_order", "contribution_direction", "rank"]
    ).reset_index(drop=True)
    ordered_columns = [
        *CELL_COLUMNS,
        "contribution_direction",
        "rank",
        "collocate",
        "collocate_type",
        "same_target_vocabulary_collocate",
        "other_target_vocabulary_collocate",
        "occurrences",
        "documents",
        value_column,
        contribution_column,
        f"{value_column}_contribution_to_cell_mean",
        "match_share",
        "total_matches_for_cell",
        "total_documents_for_cell",
        f"{value_column}_mean_for_cell",
    ]
    return output[ordered_columns]


sentiment_top = top_signed_contributors(
    collocate_contributions,
    value_column="valence",
    contribution_column="weighted_valence_contribution",
    positive_label="positive",
    negative_label="negative",
)
arousal_top = top_signed_contributors(
    collocate_contributions,
    value_column="arousal",
    contribution_column="weighted_arousal_contribution",
    positive_label="arousal_raising",
    negative_label="arousal_lowering",
)

sentiment_paths = write_table(sentiment_top, "lsc_posthoc_sentiment_collocates.csv")
arousal_paths = write_table(arousal_top, "lsc_posthoc_arousal_collocates.csv")
cell_total_paths = write_table(
    cell_totals.sort_values(["analysis_unit", "frame_stratum", "period_order"]),
    "lsc_posthoc_vad_cell_totals.csv",
)

vad_summary = pd.DataFrame(
    [
        {
            "target_vad_rows": len(vad_target),
            "unique_collocates": vad_target["collocate"].nunique(),
            "target_vad_cells": len(observed_vad_cells),
            "expected_cells": len(expected_cells),
        }
    ]
)
display(vad_summary)
display(sentiment_top.head(12))
display(arousal_top.head(12))
print("Saved VAD contributor tables:")
for output_path in [*sentiment_paths, *arousal_paths, *cell_total_paths]:
    print(f"- {output_path.relative_to(PROJECT_ROOT)}")


,target_vad_rows,unique_collocates,target_vad_cells,expected_cells
0,376759,8856,18,18


,analysis_unit,frame_stratum,frame_label,period,period_order,period_start,period_end,contribution_direction,rank,collocate,collocate_type,same_target_vocabulary_collocate,other_target_vocabulary_collocate,occurrences,documents,valence,weighted_valence_contribution,valence_contribution_to_cell_mean,match_share,total_matches_for_cell,total_documents_for_cell,valence_mean_for_cell
0,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,negative,1,disorder,unigram,False,False,383,336,-0.675000,-258.5250,-0.012138,0.017982,21299,4203,0.074155
1,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,negative,2,autism,unigram,False,True,485,443,-0.530000,-257.0500,-0.012069,0.022771,21299,4203,0.074155
2,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,negative,3,depression,unigram,False,False,237,223,-0.938000,-222.3060,-0.010437,0.011127,21299,4203,0.074155
3,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,negative,4,diagnose,unigram,False,False,355,322,-0.455500,-161.7025,-0.007592,0.016667,21299,4203,0.074155
4,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,negative,5,anxiety,unigram,False,False,215,204,-0.708000,-152.2200,-0.007147,0.010094,21299,4203,0.074155
5,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,positive,1,child,unigram,False,False,820,671,0.769000,630.5800,0.029606,0.038499,21299,4203,0.074155
6,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,positive,2,adult,unigram,False,False,268,203,0.630000,168.8400,0.007927,0.012583,21299,4203,0.074155
7,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,positive,3,add,unigram,True,False,549,464,0.290000,159.2100,0.007475,0.025776,21299,4203,0.074155
8,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,positive,4,like,unigram,False,False,162,153,0.772250,125.1045,0.005874,0.007606,21299,4203,0.074155
9,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,positive,5,include,unigram,False,False,213,207,0.459333,97.8380,0.004594,0.010000,21299,4203,0.074155


,analysis_unit,frame_stratum,frame_label,period,period_order,period_start,period_end,contribution_direction,rank,collocate,collocate_type,same_target_vocabulary_collocate,other_target_vocabulary_collocate,occurrences,documents,arousal,weighted_arousal_contribution,arousal_contribution_to_cell_mean,match_share,total_matches_for_cell,total_documents_for_cell,arousal_mean_for_cell
0,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_lowering,1,add,unigram,True,False,549,464,-0.105000,-57.645000,-0.002706,0.025776,21299,4203,0.020160
1,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_lowering,2,treatment,unigram,False,False,214,185,-0.228000,-48.792000,-0.002291,0.010047,21299,4203,0.020160
2,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_lowering,3,patient,unigram,False,False,74,68,-0.614000,-45.436000,-0.002133,0.003474,21299,4203,0.020160
3,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_lowering,4,student,unigram,False,False,116,91,-0.364000,-42.224000,-0.001982,0.005446,21299,4203,0.020160
4,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_lowering,5,common,unigram,False,False,53,53,-0.746000,-39.538000,-0.001856,0.002488,21299,4203,0.020160
5,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_raising,1,disorder,unigram,False,False,383,336,0.530000,202.990000,0.009530,0.017982,21299,4203,0.020160
6,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_raising,2,anxiety,unigram,False,False,215,204,0.730000,156.950000,0.007369,0.010094,21299,4203,0.020160
7,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_raising,3,hyperactivity,unigram,True,False,94,90,1.000000,94.000000,0.004413,0.004413,21299,4203,0.020160
8,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_raising,4,drug,unigram,False,False,123,109,0.542000,66.666000,0.003130,0.005775,21299,4203,0.020160
9,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,arousal_raising,5,suffer,unigram,False,False,97,93,0.629000,61.013000,0.002865,0.004554,21299,4203,0.020160


Saved VAD contributor tables:
- data/processed/lsc/posthoc/lsc_posthoc_sentiment_collocates.csv
- reports/tables/lsc/posthoc/lsc_posthoc_sentiment_collocates.csv
- data/processed/lsc/posthoc/lsc_posthoc_arousal_collocates.csv
- reports/tables/lsc/posthoc/lsc_posthoc_arousal_collocates.csv
- data/processed/lsc/posthoc/lsc_posthoc_vad_cell_totals.csv
- reports/tables/lsc/posthoc/lsc_posthoc_vad_cell_totals.csv


## Collocate Contributor LaTeX Tables

The tables show the top three terms from each saved top-five direction list to keep appendix candidates legible. The CSV tables retain all top-five contributors.


In [3]:
def joined_terms(frame: pd.DataFrame, unit: str, frame_stratum: str, period: str, direction: str, n: int = REPORT_TOP_TERMS) -> str:
    subset = frame.loc[
        frame["analysis_unit"].eq(unit)
        & frame["frame_stratum"].eq(frame_stratum)
        & frame["period"].eq(period)
        & frame["contribution_direction"].eq(direction)
    ].sort_values("rank")
    terms = subset["collocate"].head(n).tolist()
    return ", ".join(str(term) for term in terms) if terms else "-"


def make_contributor_latex_table(
    frame: pd.DataFrame,
    caption: str,
    label: str,
    first_direction: str,
    first_label: str,
    second_direction: str,
    second_label: str,
    note: str,
) -> str:
    rows: list[str] = []
    for unit in TARGET_UNITS:
        for frame_stratum in REPORT_FRAMES:
            period_cells = []
            for period in PERIOD_LABELS:
                first_terms = joined_terms(frame, unit, frame_stratum, period, first_direction)
                second_terms = joined_terms(frame, unit, frame_stratum, period, second_direction)
                period_cells.append(latex_cell_lines([f"{first_label}: {first_terms}", f"{second_label}: {second_terms}"]))
            rows.append(
                f"{latex_escape(unit)} & {latex_escape(FRAME_LABELS[frame_stratum])} & "
                + " & ".join(period_cells)
                + r" \\" 
            )
    return "\n".join(
        [
            r"\begin{table}[htbp]",
            r"\centering",
            f"\\caption{{{latex_escape(caption)}}}",
            f"\\label{{{label}}}",
            r"\scriptsize",
            r"\setlength{\tabcolsep}{3.5pt}",
            r"\renewcommand{\arraystretch}{1.2}",
            r"\begin{tabular}{@{}llp{0.235\linewidth}p{0.235\linewidth}p{0.235\linewidth}@{}}",
            r"\toprule",
            "Target & Frame & " + " & ".join(latex_escape(period) for period in PERIOD_LABELS) + ' \\\\',
            r"\midrule",
            *rows,
            r"\bottomrule",
            r"\end{tabular}",
            f"\\parbox{{\\linewidth}}{{\\footnotesize Note. {latex_escape(note)}}}",
            r"\end{table}",
        ]
    )


sentiment_latex_path = write_latex_table(
    make_contributor_latex_table(
        sentiment_top,
        caption="Post-hoc sentiment collocate contributors by target, frame, and period.",
        label="tab:lsc-posthoc-sentiment-contributors",
        first_direction="positive",
        first_label="Positive",
        second_direction="negative",
        second_label="Negative",
        note=(
            "Cells show the three largest preprocessed collocate contributors from each direction. "
            "The CSV export retains the top five contributors per direction and includes occurrence counts, document counts, "
            "mean valence, contribution values, and match shares."
        ),
    ),
    SENTIMENT_LATEX_PATH,
)
arousal_latex_path = write_latex_table(
    make_contributor_latex_table(
        arousal_top,
        caption="Post-hoc arousal collocate contributors by target, frame, and period.",
        label="tab:lsc-posthoc-arousal-contributors",
        first_direction="arousal_raising",
        first_label="Raising",
        second_direction="arousal_lowering",
        second_label="Lowering",
        note=(
            "Cells show the three largest preprocessed collocate contributors from each direction. "
            "The CSV export retains the top five contributors per direction and includes occurrence counts, document counts, "
            "mean arousal, contribution values, and match shares."
        ),
    ),
    AROUSAL_LATEX_PATH,
)

print("Saved collocate contributor LaTeX tables:")
for output_path in [sentiment_latex_path, arousal_latex_path]:
    print(f"- {output_path.relative_to(PROJECT_ROOT)}")


Saved collocate contributor LaTeX tables:
- reports/tables/lsc/posthoc/lsc_posthoc_sentiment_collocates.tex
- reports/tables/lsc/posthoc/lsc_posthoc_arousal_collocates.tex


## Breadth Context Contributors

Breadth is not a lexical weighted-average measure, so there is no exact collocate analogue. The closest diagnostic is context-level contribution to dispersion: for each target/frame/period cell, this notebook computes each context's average cosine distance to all other contexts in the same cell. High-distance contexts are interpreted as breadth-expanding examples.

In [4]:
def mean_pairwise_cosine_distance(normalised_vectors: np.ndarray) -> float:
    n = normalised_vectors.shape[0]
    if n < 2:
        return float("nan")
    sum_vector = normalised_vectors.sum(axis=0, dtype=np.float64)
    sum_pairwise_similarity = (float(np.dot(sum_vector, sum_vector)) - n) / 2.0
    mean_similarity = 2.0 * sum_pairwise_similarity / (n * (n - 1))
    return float(1.0 - mean_similarity)


def average_distance_to_cell(normalised_vectors: np.ndarray) -> np.ndarray:
    n = normalised_vectors.shape[0]
    if n < 2:
        return np.full(n, np.nan, dtype=float)
    sum_vector = normalised_vectors.sum(axis=0, dtype=np.float64)
    similarity_to_sum = normalised_vectors @ sum_vector
    return 1.0 - ((similarity_to_sum - 1.0) / (n - 1))


def compact_snippet(text: object, width: int = 260) -> str:
    value = re.sub(r"\s+", " ", str(text or "")).strip()
    if len(value) <= width:
        return value
    marker_start = value.find("<t>")
    marker_end = value.find("</t>")
    if marker_start >= 0 and marker_end > marker_start:
        centre = (marker_start + marker_end) // 2
    else:
        centre = len(value) // 2
    left = max(0, centre - width // 2)
    right = min(len(value), left + width)
    left = max(0, right - width)
    snippet = value[left:right].strip()
    if left > 0:
        snippet = "... " + snippet
    if right < len(value):
        snippet = snippet + " ..."
    return snippet


def content_lemmas_for_text(text: object, analysis_unit: str | None = None) -> list[str]:
    cleaned = re.sub(r"</?t>", " ", str(text or ""))
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    if not cleaned:
        return []
    excluded = set(BOILERPLATE_WORDS)
    if analysis_unit in TARGET_WORDS:
        excluded.update(TARGET_WORDS[analysis_unit])
    lemmas: list[str] = []
    for token in nlp(cleaned):
        lemma = token.lemma_.lower().strip()
        if token.is_space or token.is_punct or token.like_num or len(lemma) <= 2:
            continue
        if token.is_stop or lemma in nlp.Defaults.stop_words or lemma in excluded:
            continue
        if token.pos_ not in CONTENT_POS:
            continue
        if not re.search(r"[a-z]", lemma):
            continue
        lemmas.append(lemma)
    return lemmas


def top_unique_words(text: object, analysis_unit: str, n: int = 8) -> str:
    counts = Counter(content_lemmas_for_text(text, analysis_unit))
    return ", ".join(word for word, _ in counts.most_common(n))


breadth_contexts = pd.read_parquet(BREADTH_CONTEXT_PATH)
breadth_contexts = breadth_contexts.loc[
    breadth_contexts["analysis_unit"].isin(TARGET_UNITS)
    & breadth_contexts["frame_stratum"].isin(REPORT_FRAMES)
].copy()
breadth_contexts = add_display_order(add_period_columns(breadth_contexts))
breadth_contexts["sample_row_id"] = breadth_contexts["sample_row_id"].astype(int)
breadth_contexts["embedding_row_id"] = breadth_contexts["embedding_row_id"].astype(int)

observed_breadth_cells = set(
    breadth_contexts[["analysis_unit", "frame_stratum", "period"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
missing_breadth_cells = sorted(expected_cells - observed_breadth_cells)
if missing_breadth_cells:
    raise ValueError(f"Missing breadth cells: {missing_breadth_cells}")

embeddings_normalised = np.load(BREADTH_EMBEDDING_PATH, mmap_mode="r")
if breadth_contexts["embedding_row_id"].max() >= embeddings_normalised.shape[0]:
    raise ValueError("Breadth context embedding_row_id exceeds saved embedding matrix rows.")

context_records: list[pd.DataFrame] = []
word_source_records: list[pd.DataFrame] = []
cell_diagnostics: list[dict[str, object]] = []

for group_values, group in breadth_contexts.groupby(CELL_COLUMNS, sort=True):
    group = group.copy().reset_index(drop=True)
    vectors = np.asarray(embeddings_normalised[group["embedding_row_id"].to_numpy(dtype=int)], dtype=np.float32)
    average_distances = average_distance_to_cell(vectors)
    cell_breadth = mean_pairwise_cosine_distance(vectors)
    group["avg_distance_to_cell_contexts"] = average_distances
    group["distance_percentile_in_cell"] = group["avg_distance_to_cell_contexts"].rank(method="average", pct=True)
    std = group["avg_distance_to_cell_contexts"].std(ddof=0)
    group["distance_z_in_cell"] = 0.0 if pd.isna(std) or std == 0 else (
        group["avg_distance_to_cell_contexts"] - group["avg_distance_to_cell_contexts"].mean()
    ) / std
    group["cell_breadth_mean_pairwise_distance"] = cell_breadth
    group["cell_contexts"] = len(group)
    group["cell_documents"] = group["doc_id"].nunique()
    group["cell_domains"] = group["registered_domain"].nunique()

    top_contexts = (
        group.sort_values("avg_distance_to_cell_contexts", ascending=False)
        .head(BREADTH_TOP_CONTEXTS_PER_CELL)
        .copy()
    )
    top_contexts["rank"] = np.arange(1, len(top_contexts) + 1)
    top_contexts["snippet"] = top_contexts["marked_context"].map(compact_snippet)
    top_contexts["content_word_hints"] = [
        top_unique_words(text, unit)
        for text, unit in zip(top_contexts["marked_context"], top_contexts["analysis_unit"], strict=True)
    ]
    context_records.append(top_contexts)

    word_source = (
        group.sort_values("avg_distance_to_cell_contexts", ascending=False)
        .head(BREADTH_WORD_SOURCE_CONTEXTS_PER_CELL)
        .copy()
    )
    word_source_records.append(word_source)

    cell_diagnostics.append(
        {
            "analysis_unit": group_values[0],
            "frame_stratum": group_values[1],
            "frame_label": group_values[2],
            "period": group_values[3],
            "period_order": group_values[4],
            "period_start": group_values[5],
            "period_end": group_values[6],
            "cell_contexts": len(group),
            "cell_documents": group["doc_id"].nunique(),
            "cell_domains": group["registered_domain"].nunique(),
            "cell_breadth_mean_pairwise_distance": cell_breadth,
            "mean_context_avg_distance": float(np.nanmean(average_distances)),
            "max_context_avg_distance": float(np.nanmax(average_distances)),
        }
    )

breadth_context_contributors = pd.concat(context_records, ignore_index=True)
breadth_word_source = pd.concat(word_source_records, ignore_index=True)
breadth_cell_diagnostics = pd.DataFrame(cell_diagnostics)

word_records: list[dict[str, object]] = []
for row in breadth_word_source.itertuples(index=False):
    words = content_lemmas_for_text(row.marked_context, row.analysis_unit)
    for word in words:
        word_records.append(
            {
                "analysis_unit": row.analysis_unit,
                "frame_stratum": row.frame_stratum,
                "frame_label": row.frame_label,
                "period": row.period,
                "period_order": row.period_order,
                "period_start": row.period_start,
                "period_end": row.period_end,
                "word": word,
                "sample_row_id": row.sample_row_id,
                "doc_id": row.doc_id,
                "avg_distance_to_cell_contexts": row.avg_distance_to_cell_contexts,
            }
        )

breadth_word_summary = pd.DataFrame(word_records)
if breadth_word_summary.empty:
    raise ValueError("No content words were extracted from high-distance breadth contexts.")
word_cell_counts = (
    breadth_word_source.groupby(CELL_COLUMNS, as_index=False)
    .agg(high_distance_contexts_used=("sample_row_id", "nunique"))
)
breadth_word_summary = (
    breadth_word_summary.groupby([*CELL_COLUMNS, "word"], as_index=False)
    .agg(
        occurrences_in_high_distance_contexts=("word", "size"),
        high_distance_contexts_with_word=("sample_row_id", "nunique"),
        high_distance_documents_with_word=("doc_id", "nunique"),
        mean_context_avg_distance=("avg_distance_to_cell_contexts", "mean"),
    )
    .merge(word_cell_counts, on=CELL_COLUMNS, how="left")
)
breadth_word_summary["context_share"] = (
    breadth_word_summary["high_distance_contexts_with_word"]
    / breadth_word_summary["high_distance_contexts_used"]
)
breadth_word_summary = add_display_order(breadth_word_summary)
breadth_word_summary = (
    breadth_word_summary.sort_values(
        [
            "analysis_order",
            "frame_order",
            "period_order",
            "high_distance_contexts_with_word",
            "occurrences_in_high_distance_contexts",
            "word",
        ],
        ascending=[True, True, True, False, False, True],
    )
    .groupby(CELL_COLUMNS, as_index=False)
    .head(BREADTH_TOP_WORDS_PER_CELL)
    .reset_index(drop=True)
)
breadth_word_summary["rank"] = breadth_word_summary.groupby(CELL_COLUMNS).cumcount() + 1

context_columns = [
    *CELL_COLUMNS,
    "rank",
    "lsc_year",
    "doc_id",
    "registered_domain",
    "raw_form",
    "context_source",
    "context_token_count",
    "avg_distance_to_cell_contexts",
    "distance_percentile_in_cell",
    "distance_z_in_cell",
    "cell_breadth_mean_pairwise_distance",
    "cell_contexts",
    "cell_documents",
    "cell_domains",
    "content_word_hints",
    "snippet",
]
breadth_context_contributors = (
    add_display_order(breadth_context_contributors)
    .sort_values(["analysis_order", "frame_order", "period_order", "rank"])
    .reset_index(drop=True)
)
breadth_context_contributors = breadth_context_contributors[context_columns]

word_columns = [
    *CELL_COLUMNS,
    "rank",
    "word",
    "occurrences_in_high_distance_contexts",
    "high_distance_contexts_with_word",
    "high_distance_documents_with_word",
    "high_distance_contexts_used",
    "context_share",
    "mean_context_avg_distance",
]
breadth_word_summary = breadth_word_summary[word_columns]
breadth_cell_diagnostics = breadth_cell_diagnostics.sort_values(["analysis_unit", "frame_stratum", "period_order"])

breadth_context_paths = write_table(
    breadth_context_contributors,
    "lsc_posthoc_breadth_context_contributors.csv",
)
breadth_word_paths = write_table(
    breadth_word_summary,
    "lsc_posthoc_breadth_content_word_summaries.csv",
)
breadth_diagnostic_paths = write_table(
    breadth_cell_diagnostics,
    "lsc_posthoc_breadth_cell_diagnostics.csv",
)

display(breadth_cell_diagnostics.head(9))
display(breadth_context_contributors.head(12))
display(breadth_word_summary.head(12))
print("Saved breadth contributor tables:")
for output_path in [*breadth_context_paths, *breadth_word_paths, *breadth_diagnostic_paths]:
    print(f"- {output_path.relative_to(PROJECT_ROOT)}")

,analysis_unit,frame_stratum,frame_label,period,period_order,period_start,period_end,cell_contexts,cell_documents,cell_domains,cell_breadth_mean_pairwise_distance,mean_context_avg_distance,max_context_avg_distance
0,ADHD,clinical_only,Clinical,2014-2017,1,2014,2017,4495,3123,2340,0.150293,0.150293,0.591353
1,ADHD,clinical_only,Clinical,2018-2021,2,2018,2021,3886,2765,2450,0.139985,0.139985,0.522984
2,ADHD,clinical_only,Clinical,2022-2026,3,2022,2026,3073,2118,1916,0.135732,0.135732,0.441109
3,ADHD,lived_only,Lived experience,2014-2017,1,2014,2017,1085,898,771,0.078423,0.078423,0.407867
4,ADHD,lived_only,Lived experience,2018-2021,2,2018,2021,1124,911,823,0.082683,0.082683,0.509366
5,ADHD,lived_only,Lived experience,2022-2026,3,2022,2026,1194,921,847,0.075956,0.075956,0.569576
6,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,6220,4254,3177,0.136908,0.136908,0.596653
7,ADHD,substantive_core_overall,Overall,2018-2021,2,2018,2021,5626,3937,3426,0.127375,0.127375,0.525702
8,ADHD,substantive_core_overall,Overall,2022-2026,3,2022,2026,4883,3253,2897,0.119728,0.119728,0.589165


,analysis_unit,frame_stratum,frame_label,period,period_order,period_start,period_end,rank,lsc_year,doc_id,registered_domain,raw_form,context_source,context_token_count,avg_distance_to_cell_contexts,distance_percentile_in_cell,distance_z_in_cell,cell_breadth_mean_pairwise_distance,cell_contexts,cell_documents,cell_domains,content_word_hints,snippet
0,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,1,2016,36367539aa8db7be,kuscco.com,adhd,target_sentence,33,0.596653,1.000000,7.285519,0.136908,6220,4254,3177,"high, recovered, dairy, prior, liquid, acu, rite, meridia","Although a higher <t>adhd</t> of the recovered dairy from prior others was liquid to 4 acu-rite; meridia of malaria at the online reduction discoloration, this does properly in..."
1,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,2,2016,80b8d3c69882904e,criminal-record-check.life,adhd,target_sentence,49,0.467933,0.999839,5.245700,0.136908,6220,4254,3177,"tell, prince, likely, possess, bed, census, deliver, mountain","He tells the prince's <t>adhd</t> that he is the likely one who possesses his bed's census, delivered on his mountain check, of casting a franchise bone and persuades them to t..."
2,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,3,2015,2ac9fd2ba3f1f4a0,illinoistimes.com,adhd,target_sentence,28,0.432776,0.999598,4.688571,0.136908,6220,4254,3177,"production, form, productions, local, actor, jason, goodreau, mac","It’s the first production of the just-formed <t>ADHD</t> Productions, and local actors Jason Goodreau and Mac Warren are performing this must-see play by Richard Dresser."
3,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,4,2014,5a02a2c8b3617909,illinoistimes.com,adhd,target_sentence,28,0.432776,0.999598,4.688571,0.136908,6220,4254,3177,"production, form, productions, local, actor, jason, goodreau, mac","It’s the first production of the just-formed <t>ADHD</t> Productions, and local actors Jason Goodreau and Mac Warren are performing this must-see play by Richard Dresser."
4,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,5,2017,7203541365cd664e,additudemag.com,adhd,target_sentence,73,0.420999,0.999357,4.501950,0.136908,6220,4254,3177,"text, buy, thing, middle, world, completely, google, ads",To me the world itself seems to have gone completely <t>ADHD</t> whenever: - Google Ads suggest I buy things I just bought - The phone rings in the middle of a text conversatio...
5,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,6,2014,85efe696f8e2bc12,thewrap.com,adhd,target_sentence,21,0.413120,0.999196,4.377087,0.136908,6220,4254,3177,"network, order, traditional, half, hour, studio, premiere, primetime","The network has also ordered two traditional half-hour shows from the <t>ADHD</t> studio, which will premiere in primetime in 2015."
6,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,7,2014,612af73ee3c56e54,wikia.com,adhd,target_sentence,24,0.413117,0.999035,4.377050,0.136908,6220,4254,3177,"plugin, edit, architecture, great, joy, acrobat, star, splash",edit Plugin Architecture One of the great joys of Acrobat is staring at the splash screen waiting for countless plugins to load (see <t>ADHD</t>).
7,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,8,2017,8257571c03571573,theinscribermag.com,adhd,target_sentence,10,0.412968,0.998875,4.374685,0.136908,6220,4254,3177,"problem, company","However, problem has always been that this company has <t>ADHD</t>."
8,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,9,2016,76d4d3f5e6a0509f,healthyplace.com,adhd,target_sentence,23,0.409417,0.998714,4.318401,0.136908,6220,4254,3177,"article, different, types, educational, assessment, tests, parent, advocate","next: Different Types of Educational Assessment Tests ~ back to Parent Advocate homepage ~ <t>adhd</t> library articles ~ all add/adhd articles APA Reference Staff, H."
9,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,10,2014,6b893a4bd106a0a4,iss

,analysis_unit,frame_stratum,frame_label,period,period_order,period_start,period_end,rank,word,occurrences_in_high_distance_contexts,high_distance_contexts_with_word,high_distance_documents_with_word,high_distance_contexts_used,context_share,mean_context_avg_distance
0,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,1,disorder,25,10,9,20,0.50,0.381292
1,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,2,behavior,4,4,4,20,0.20,0.377688
2,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,3,childhood,4,4,4,20,0.20,0.380967
3,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,4,impulsivity,4,4,4,20,0.20,0.380967
4,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,5,inattention,4,4,4,20,0.20,0.380967
5,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,6,case,3,3,3,20,0.15,0.377204
6,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,7,characterize,3,3,3,20,0.15,0.377204
7,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,8,diagnose,3,3,3,20,0.15,0.377204
8,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,9,problem,3,3,3,20,0.15,0.393560
9,ADHD,substantive_core_overall,Overall,2014-2017,1,2014,2017,10,spectrum,3,3,2,20,0.15,0.383422


Saved breadth contributor tables:
- data/processed/lsc/posthoc/lsc_posthoc_breadth_context_contributors.csv
- reports/tables/lsc/posthoc/lsc_posthoc_breadth_context_contributors.csv
- data/processed/lsc/posthoc/lsc_posthoc_breadth_content_word_summaries.csv
- reports/tables/lsc/posthoc/lsc_posthoc_breadth_content_word_summaries.csv
- data/processed/lsc/posthoc/lsc_posthoc_breadth_cell_diagnostics.csv
- reports/tables/lsc/posthoc/lsc_posthoc_breadth_cell_diagnostics.csv


## Breadth Context Content-Word Table

The breadth CSV outputs contain high-distance context snippets and filtered content words from those contexts. This is a context diagnostic, not a raw collocate table. The compact LaTeX table shows the top five high-distance content words per target/frame/period cell.


In [5]:
def joined_breadth_words(unit: str, frame_stratum: str, period: str, n: int = 5) -> str:
    subset = breadth_word_summary.loc[
        breadth_word_summary["analysis_unit"].eq(unit)
        & breadth_word_summary["frame_stratum"].eq(frame_stratum)
        & breadth_word_summary["period"].eq(period)
    ].sort_values("rank")
    words = subset["word"].head(n).tolist()
    return ", ".join(str(word) for word in words) if words else "-"


def make_breadth_latex_table() -> str:
    rows: list[str] = []
    for unit in TARGET_UNITS:
        for frame_stratum in REPORT_FRAMES:
            period_cells = [latex_escape(joined_breadth_words(unit, frame_stratum, period)) for period in PERIOD_LABELS]
            rows.append(
                f"{latex_escape(unit)} & {latex_escape(FRAME_LABELS[frame_stratum])} & "
                + " & ".join(period_cells)
                + r" \\" 
            )
    note = (
        "Cells show the five most frequent filtered content lemmas among the twenty highest-distance contexts in each cell. "
        "This breadth diagnostic summarises high-distance context wording rather than VAD collocates; the CSV exports retain context-level contributors, word-level counts, and cell diagnostics."
    )
    return "\n".join(
        [
            r"\begin{table}[htbp]",
            r"\centering",
            r"\caption{Post-hoc breadth high-distance content words by target, frame, and period.}",
            r"\label{tab:lsc-posthoc-breadth-contributors}",
            r"\scriptsize",
            r"\setlength{\tabcolsep}{3.5pt}",
            r"\renewcommand{\arraystretch}{1.25}",
            r"\begin{tabular}{@{}llp{0.235\linewidth}p{0.235\linewidth}p{0.235\linewidth}@{}}",
            r"\toprule",
            "Target & Frame & " + " & ".join(latex_escape(period) for period in PERIOD_LABELS) + ' \\\\',
            r"\midrule",
            *rows,
            r"\bottomrule",
            r"\end{tabular}",
            f"\\parbox{{\\linewidth}}{{\\footnotesize Note. {latex_escape(note)}}}",
            r"\end{table}",
        ]
    )


breadth_latex_path = write_latex_table(make_breadth_latex_table(), BREADTH_WORD_LATEX_PATH)

print("Saved breadth contributor LaTeX table:")
print(f"- {breadth_latex_path.relative_to(PROJECT_ROOT)}")


Saved breadth contributor LaTeX table:
- reports/tables/lsc/posthoc/lsc_posthoc_breadth_content_words.tex


## Validation And Handoff

The checks below confirm that all target/frame/period cells are represented and that the post-hoc outputs are aligned with the completed annual analyses.

In [6]:
sentiment_annual = pd.read_csv(SENTIMENT_ANNUAL_PATH)
intensity_annual = pd.read_csv(INTENSITY_ANNUAL_PATH)
breadth_annual = pd.read_csv(BREADTH_ANNUAL_PATH)

for name, frame in [
    ("sentiment annual", sentiment_annual),
    ("intensity annual", intensity_annual),
    ("breadth annual", breadth_annual),
]:
    observed_years = sorted(
        frame.loc[
            frame["analysis_unit"].isin(TARGET_UNITS)
            & frame["frame_stratum"].isin(REPORT_FRAMES),
            "lsc_year",
        ].dropna().unique().astype(int).tolist()
    )
    if observed_years != list(range(2014, 2027)):
        raise ValueError(f"Unexpected year coverage for {name}: {observed_years}")

for latex_path in [sentiment_latex_path, arousal_latex_path, breadth_latex_path]:
    if not latex_path.exists() or not latex_path.read_text(encoding="utf-8").strip():
        raise ValueError(f"Missing or empty LaTeX output: {latex_path}")

output_checks = {
    "sentiment_top_rows": len(sentiment_top),
    "arousal_top_rows": len(arousal_top),
    "breadth_context_rows": len(breadth_context_contributors),
    "breadth_word_rows": len(breadth_word_summary),
    "vad_cells": len(observed_vad_cells),
    "breadth_cells": len(observed_breadth_cells),
    "expected_cells": len(expected_cells),
}
if output_checks["vad_cells"] != output_checks["expected_cells"]:
    raise ValueError(output_checks)
if output_checks["breadth_cells"] != output_checks["expected_cells"]:
    raise ValueError(output_checks)

handoff = pd.DataFrame(
    [
        {"artifact": "sentiment collocates", "rows": len(sentiment_top), "path": sentiment_paths[0].relative_to(PROJECT_ROOT)},
        {"artifact": "sentiment contributor LaTeX", "rows": len(TARGET_UNITS) * len(REPORT_FRAMES), "path": sentiment_latex_path.relative_to(PROJECT_ROOT)},
        {"artifact": "arousal collocates", "rows": len(arousal_top), "path": arousal_paths[0].relative_to(PROJECT_ROOT)},
        {"artifact": "arousal contributor LaTeX", "rows": len(TARGET_UNITS) * len(REPORT_FRAMES), "path": arousal_latex_path.relative_to(PROJECT_ROOT)},
        {"artifact": "VAD cell totals", "rows": len(cell_totals), "path": cell_total_paths[0].relative_to(PROJECT_ROOT)},
        {"artifact": "breadth context contributors", "rows": len(breadth_context_contributors), "path": breadth_context_paths[0].relative_to(PROJECT_ROOT)},
        {"artifact": "breadth content word summaries", "rows": len(breadth_word_summary), "path": breadth_word_paths[0].relative_to(PROJECT_ROOT)},
        {"artifact": "breadth content-word LaTeX", "rows": len(TARGET_UNITS) * len(REPORT_FRAMES), "path": breadth_latex_path.relative_to(PROJECT_ROOT)},
        {"artifact": "breadth cell diagnostics", "rows": len(breadth_cell_diagnostics), "path": breadth_diagnostic_paths[0].relative_to(PROJECT_ROOT)},
    ]
)
display(handoff)
print("Validation checks passed.")


,artifact,rows,path
0,sentiment collocates,180,data/processed/lsc/posthoc/lsc_posthoc_sentiment_collocates.csv
1,sentiment contributor LaTeX,6,reports/tables/lsc/posthoc/lsc_posthoc_sentiment_collocates.tex
2,arousal collocates,180,data/processed/lsc/posthoc/lsc_posthoc_arousal_collocates.csv
3,arousal contributor LaTeX,6,reports/tables/lsc/posthoc/lsc_posthoc_arousal_collocates.tex
4,VAD cell totals,18,data/processed/lsc/posthoc/lsc_posthoc_vad_cell_totals.csv
5,breadth context contributors,180,data/processed/lsc/posthoc/lsc_posthoc_breadth_context_contributors.csv
6,breadth content word summaries,180,data/processed/lsc/posthoc/lsc_posthoc_breadth_content_word_summaries.csv
7,breadth content-word LaTeX,6,reports/tables/lsc/posthoc/lsc_posthoc_breadth_content_words.tex
8,breadth cell diagnostics,18,data/processed/lsc/posthoc/lsc_posthoc_breadth_cell_diagnostics.csv


Validation checks passed.
